# บทเรียนที่ 13 - ความทรงจำของตัวแทน


## การตั้งค่า

สมุดบันทึกนี้แสดงวิธีสร้างตัวแทนจองการเดินทางที่มี **หน่วยความจำถาวร** โดยใช้ **Microsoft Agent Framework** (MAF)

คุณจะได้เรียนรู้ว่าวิธีที่หน่วยความจำของตัวแทนประเภทต่าง ๆ — หน่วยความจำการทำงาน, หน่วยความจำระยะสั้น และหน่วยความจำระยะยาว — มีผลต่อวิธีที่ตัวแทนเก็บรักษาและใช้ข้อมูลข้ามการสนทนาอย่างไร

**ข้อกำหนดเบื้องต้น:**
- โครงการ Azure AI Foundry ที่มีโมเดลแชทที่ถูกปรับใช้แล้ว (เช่น `gpt-4o-mini`)
- เข้าสู่ระบบด้วย Azure CLI — รัน `az login` ในเทอร์มินัลของคุณ
- `AZURE_AI_PROJECT_ENDPOINT` — จุดสิ้นสุดของโครงการ Azure AI Foundry ของคุณ
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ชื่อของโมเดลที่คุณได้ปรับใช้แล้ว


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ประเภทของความจำของเอเย่นต์

เอเย่นต์ AI สามารถใช้ความจำประเภทต่าง ๆ ซึ่งแต่ละประเภทมีวัตถุประสงค์ที่แตกต่างกัน:

### ความจำในการทำงาน
กระทู้การสนทนาเอง — ข้อความที่แลกเปลี่ยนกันในเซสชันเดียว เอเย่นต์สามารถอ้างอิงกลับไปยังข้อความก่อนหน้าในกระทู้เดียวกันเพื่อรักษาความสอดคล้อง ใน MAF นี้ถูกสร้างผ่าน **`agent.create_session()`** ซึ่งจะคืนค่า `AgentSession`

### ความจำระยะสั้น
ข้อมูลที่คงอยู่ในระหว่างการทำงานหรือเซสชันแต่ไม่ถูกเก็บถาวร ตัวอย่างเช่น เอเย่นต์อาจสะสมข้อเท็จจริงระหว่างการสนทนาแบบหลายรอบและใช้ข้อมูลเหล่านั้นเพื่อสร้างแผนการเดินทางสุดท้าย

### ความจำระยะยาว
ความชอบและข้อเท็จจริงที่คงอยู่ **ข้ามเซสชัน** ผู้ใช้ที่กลับมาไม่ควรต้องบอกข้อจำกัดทางอาหารหรือรูปแบบการเดินทางซ้ำ ความจำระยะยาวมักถูกสนับสนุนโดยแหล่งเก็บข้อมูลภายนอก — ฐานข้อมูล ไฟล์ หรือดัชนีเวกเตอร์ — และนำเสนอแก่เอเย่นต์ผ่านเครื่องมือ


## Working Memory with Sessions

รูปแบบหน่วยความจำที่ง่ายที่สุดคือเซสชันการสนทนา เมื่อคุณส่งต่อเซสชันเดียวกัน (ซึ่งสร้างขึ้นผ่าน `agent.create_session()`) ไปยังการเรียก `agent.run()` ต่อเนื่อง ตัวแทนจะเห็นประวัติการสนทนาทั้งหมดและสามารถจดจำรายละเอียดก่อนหน้าได้

มาสร้างตัวแทนท่องเที่ยวและสาธิตหน่วยความจำการทำงานกันเถอะ


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ตัวแทนจำได้งบประมาณอย่างถูกต้องเพราะทั้งสองข้อความแชร์เซสชันเดียวกัน นี่คือ **หน่วยความจำทำงาน** — ซึ่งมีอยู่เฉพาะในช่วงเวลาของเซสชันเท่านั้น

### จะเกิดอะไรขึ้นกับเธรดใหม่?

ถ้าเราสร้างเซสชัน **ใหม่** ตัวแทนจะไม่มีหน่วยความจำของการสนทนาก่อนหน้า:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## แบบจดจำระยะยาว

เพื่อจดจำความชอบของผู้ใช้ **ข้ามการสนทนา** เราจำเป็นต้องมีที่จัดเก็บข้อมูลที่คงอยู่ภายนอกเธรดการสนทนา ตัวแทนเข้าถึงที่จัดเก็บนี้ผ่านทาง **เครื่องมือ** — ฟังก์ชันที่มันสามารถเรียกใช้เพื่อบันทึกและดึงข้อมูลออกมา

ด้านล่างนี้เราได้สร้างที่จัดเก็บความชอบในหน่วยความจำอย่างง่าย (ในสภาพแวดล้อมจริงคุณจะใช้ฐานข้อมูลหรือดัชนีเวกเตอร์รองรับ) และเปิดเผยเป็นเครื่องมือให้ตัวแทนใช้

### สถาปัตยกรรม
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — ผู้ใช้ครั้งแรกจองทริปครบรอบ

ซาร่าห์มาเยือนเป็นครั้งแรก ตัวแทนควรบันทึกความชอบของเธอผ่านเครื่องมือต่างๆ และใช้ข้อมูลเหล่านั้นเพื่อแนะนำโรงแรม


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah กลับมาอีกหลายสัปดาห์ต่อมา

Sarah เริ่ม **เธรดใหม่เอี่ยม** (จำลองการเริ่มเซสชันใหม่) หน่วยความจำทำงานว่างเปล่า แต่คลังข้อมูลความชอบระยะยาวยังคงมีข้อมูลของเธอ เอเจนต์ควรดึงข้อมูลนั้นมาใช้เพื่อปรับคำแนะนำให้เหมาะกับบุคคลมากขึ้น


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้เกี่ยวกับความทรงจำของตัวแทนสามประเภทและวิธีการใช้งานกับ Microsoft Agent Framework:

| ประเภทความทรงจำ | กลไกของ MAF | ระยะเวลาใช้งาน |
|---|---|---|
| **ทำงาน** | `agent.create_session()` | การสนทนาเดียว |
| **ระยะสั้น** | สะสมบริบทภายในเธรด | งาน / เซสชันเดียว |
| **ระยะยาว** | ที่เก็บข้อมูลภายนอกเข้าถึงผ่านฟังก์ชัน `@tool` | ข้ามเซสชัน |

### ข้อสรุปสำคัญ
1. **`agent.create_session()`** ให้ความทรงจำทำงาน — ตัวแทนเห็นประวัติการสนทนาเต็มภายในเซสชัน
2. **เซสชันใหม่สูญเสียบริบท** — หากไม่มีความทรงจำระยะยาว ตัวแทนไม่สามารถจำการสนทนาในอดีตได้
3. **ฟังก์ชัน `@tool`** เป็นสะพาน — ช่วยให้ตัวแทนบันทึกและเรียกข้อมูลจากที่เก็บถาวร
4. **การปรับแต่งส่วนบุคคลดีขึ้นตามเวลา** — ยิ่งเก็บความชอบมาก การแนะนำของตัวแทนยิ่งดีขึ้น

### การใช้งานในโลกจริง
- **บริการลูกค้า**: จดจำประวัติและความชอบของลูกค้า
- **ผู้ช่วยส่วนบุคคล**: รักษาบริบทข้ามวันหรือสัปดาห์
- **การดูแลสุขภาพ**: ติดตามข้อมูลและความชอบของผู้ป่วย
- **อีคอมเมิร์ซ**: การช็อปปิ้งแบบปรับแต่งตามประวัติ

### ขั้นตอนต่อไป
- เปลี่ยน dict ในหน่วยความจำเป็นฐานข้อมูลหรือที่เก็บเวกเตอร์ (เช่น Azure AI Search)
- เพิ่มวันหมดอายุของความทรงจำสำหรับข้อมูลที่ไวต่อเวลา
- สร้างระบบหลายตัวแทนที่ใช้ความทรงจำร่วมกัน
- สำรวจโน้ตบุ๊ก Cognee สำหรับความทรงจำที่เก็บรองรับกราฟความรู้


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**คำปฏิเสธความรับผิดชอบ**:  
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาด้วย AI [Co-op Translator](https://github.com/Azure/co-op-translator) แม้ว่าเราจะพยายามทำให้การแปลมีความถูกต้อง โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้องได้ เอกสารต้นฉบับในภาษาต้นทางถือเป็นแหล่งข้อมูลที่น่าเชื่อถือ สำหรับข้อมูลที่มีความสำคัญ ควรใช้บริการแปลโดยผู้เชี่ยวชาญทางภาษามนุษย์ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
